<a href="https://colab.research.google.com/github/alvarezpablo/Llama-Tech-Talks/blob/main/Llama3.1-MetaDay-UR-2025/módulo%204/Llama3.1_Unsloth_FineTuning_Optimizado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦙 Fine-tune Llama 3.1 Ultra-Efficiently with Unsloth

## Llama 3.1 - Capacitación SICYT Argentina 2025

### Módulo 4: Fine-tuning Optimizado - Octubre 2025

Este notebook implementa las técnicas más avanzadas de fine-tuning usando **Unsloth** para obtener:
- ⚡ **2x más rápido** que métodos tradicionales
- 💾 **60% menos uso de memoria**
- 🎯 **Rank-Stabilized LoRA (rsLoRA)**
- 📊 **Chat templates optimizados**
- 🔧 **Configuración automática de hiperparámetros**

**Capacitación:** SICYT Argentina 2025  
Basado en el artículo: [Fine-tune Llama 3.1 Ultra-Efficiently with Unsloth](https://huggingface.co/blog/mlabonne/sft-llama3)

## 🚀 Configuración e Instalación

### Verificar GPU disponible

In [ ]:
import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ Ejecutando en CPU - El entrenamiento será más lento")

CUDA disponible: True
GPU: NVIDIA A100-SXM4-40GB
Memoria GPU: 39.6 GB


### Instalar Unsloth y dependencias optimizadas

In [ ]:
# 🔧 Solución para error de protobuf
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# Instalación optimizada para Colab
!pip install "protobuf<=3.20.3"
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

print("✅ Instalación completada con fix de protobuf")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-mb5ho1sn/unsloth_ac0fd75be3ec4905962cada627a23d60
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-mb5ho1sn/unsloth_ac0fd75be3ec4905962cada627a23d60
  Resolved https://github.com/unslothai/unsloth.git to commit 133d94010b2713e950bcd80078ffa4ba9b682804
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of unsloth[colab-new] to determine which version is compatible with other requirements. This could take a while.
ERROR: Ignored the following versions that require a different python version: 2025.3.2 Requires-Python <=3.12,>=3.9
ERROR: Could not find a version that satisfies the requirement ransformers!=4.47.0,!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,>=4.51.3; extra == "colab-new" (from unsloth[colab-

KeyboardInterrupt: 

### Importar librerías

In [ ]:
import torch
from trl import SFTTrainer
from datasets import load_dataset
from transformers import TrainingArguments, TextStreamer
from unsloth.chat_templates import get_chat_template
from unsloth import FastLanguageModel, is_bfloat16_supported
import os
from datetime import datetime

print(f"🚀 Configuración completada - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

/tmp/ipython-input-952506712.py:5: UserWarning: WARNING: Unsloth should be imported before trl, transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth.chat_templates import get_chat_template


Unsloth: Patching Xformers to fix some performance issues.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


    PyTorch 2.3.0+cu121 with CUDA 1201 (you have 2.6.0+cu124)
    Python  3.11.9 (you have 3.11.13)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


🦥 Unsloth Zoo will now patch everything to make training faster!
🚀 Configuración completada - 2025-08-18 04:20:45


## 🤖 Carga del Modelo Llama 3.1 8B

Usamos la versión pre-cuantizada de Unsloth para máxima eficiencia:

In [ ]:
# Configuración del modelo
max_seq_length = 2048  # Ajusta según tu GPU (hasta 128k para Llama 3.1)
model_name = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"

print(f"📥 Cargando modelo: {model_name}")
print(f"📏 Longitud máxima de secuencia: {max_seq_length}")

# Cargar modelo y tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,  # Auto-detecta BF16 para GPUs Ampere+
)

print("✅ Modelo cargado exitosamente")

📥 Cargando modelo: unsloth/Meta-Llama-3.1-8B-bnb-4bit
📏 Longitud máxima de secuencia: 2048
==((====))==  Unsloth 2025.8.6: Fast Llama patching. Transformers: 4.55.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

✅ Modelo cargado exitosamente


## ⚙️ Configuración LoRA Optimizada

Implementamos **Rank-Stabilized LoRA (rsLoRA)** con configuración optimizada:

In [ ]:
# Configuración LoRA optimizada
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Rank - balance entre calidad y eficiencia
    lora_alpha=16,  # Scaling factor (típicamente 1x o 2x el rank)
    lora_dropout=0,  # Sin dropout para entrenamiento más rápido
    target_modules=[
        "q_proj", "k_proj", "v_proj",  # Attention
        "up_proj", "down_proj", "o_proj", "gate_proj"  # Feed-forward
    ],
    use_rslora=True,  # 🎯 Rank-Stabilized LoRA para mejor estabilidad
    use_gradient_checkpointing="unsloth"  # Optimización de memoria
)

# Mostrar estadísticas de parámetros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
percentage = (trainable_params / total_params) * 100

print(f"📊 Parámetros totales: {total_params:,}")
print(f"🎯 Parámetros entrenables: {trainable_params:,} ({percentage:.4f}%)")
print(f"💾 Reducción de parámetros: {100-percentage:.2f}%")

Unsloth 2025.8.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


📊 Parámetros totales: 4,582,543,360
🎯 Parámetros entrenables: 41,943,040 (0.9153%)
💾 Reducción de parámetros: 99.08%


## 📚 Preparación del Dataset

Usamos el dataset **FineTome-100k** - ultra alta calidad con conversaciones, razonamiento y function calling:

In [ ]:
# Configurar chat template (ChatML - estándar de la comunidad)
tokenizer = get_chat_template(
    tokenizer,
    mapping={"role": "from", "content": "value", "user": "human", "assistant": "gpt"},
    chat_template="chatml",
)

def apply_template(examples):
    """Aplica el chat template a las conversaciones"""
    messages = examples["conversations"]
    text = [
        tokenizer.apply_chat_template(
            message,
            tokenize=False,
            add_generation_prompt=False
        ) for message in messages
    ]
    return {"text": text}

print("✅ Chat template configurado (ChatML)")

Unsloth: Will map <|im_end|> to EOS = <|end_of_text|>.


✅ Chat template configurado (ChatML)


In [ ]:
# Cargar dataset - ajusta el subset para entrenamientos más rápidos
dataset_size = "train[:10000]"  # Cambia a "train" para dataset completo (100k samples)

print(f"📥 Cargando dataset: mlabonne/FineTome-100k ({dataset_size})")
dataset = load_dataset("mlabonne/FineTome-100k", split=dataset_size)
dataset = dataset.map(apply_template, batched=True)

print(f"📊 Dataset cargado: {len(dataset):,} muestras")
print(f"📝 Ejemplo de conversación formateada:")
print("=" * 50)
print(dataset[0]["text"][:500] + "...")
print("=" * 50)

📥 Cargando dataset: mlabonne/FineTome-100k (train[:10000])


README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

📊 Dataset cargado: 10,000 muestras
📝 Ejemplo de conversación formateada:
<|im_start|>user
Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. 

Furthermore, add the requirement that the code must be written in a language that does not suppo...


## 🏋️ Entrenamiento con Hiperparámetros Optimizados

Configuración basada en las mejores prácticas del artículo de Hugging Face:

In [ ]:
# Configuración de entrenamiento optimizada
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=True,  # 🚀 Combina múltiples muestras pequeñas en un batch
    args=TrainingArguments(
        # Configuración de learning rate
        learning_rate=3e-4,  # Óptimo para LoRA
        lr_scheduler_type="linear",  # Scheduler lineal recomendado

        # Configuración de batch
        per_device_train_batch_size=8,  # Ajusta según tu GPU
        gradient_accumulation_steps=2,  # Batch efectivo = 8 * 2 = 16

        # Épocas y pasos
        num_train_epochs=1,  # 1 época suele ser suficiente con datasets de calidad
        warmup_steps=10,  # Warmup para estabilizar entrenamiento inicial

        # Optimización
        fp16=not is_bfloat16_supported(),  # FP16 para GPUs más antiguas
        bf16=is_bfloat16_supported(),      # BF16 para GPUs Ampere+
        optim="adamw_8bit",  # 🎯 AdamW 8-bit para menor uso de memoria
        weight_decay=0.01,   # Regularización

        # Logging y guardado
        logging_steps=1,
        output_dir="output",
        seed=0,  # Reproducibilidad

        # Configuraciones adicionales
        remove_unused_columns=False,
        dataloader_pin_memory=False,
    ),
)

print("⚙️ Trainer configurado con hiperparámetros optimizados")
print(f"🎯 Batch size efectivo: {8 * 2} (per_device_batch_size * gradient_accumulation_steps)")
print(f"🔧 Optimizador: AdamW 8-bit")
print(f"📊 Precisión: {'BF16' if is_bfloat16_supported() else 'FP16'}")

Generating train split: 0 examples [00:00, ? examples/s]

⚙️ Trainer configurado con hiperparámetros optimizados
🎯 Batch size efectivo: 16 (per_device_batch_size * gradient_accumulation_steps)
🔧 Optimizador: AdamW 8-bit
📊 Precisión: BF16


In [ ]:
# 🚀 ¡Iniciar entrenamiento!
print("🏋️ Iniciando entrenamiento...")
print(f"⏰ Tiempo estimado: ~20-30 min para 10k muestras en T4")
print("=" * 60)

trainer.train()

print("=" * 60)
print("🎉 ¡Entrenamiento completado!")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,709 | Num Epochs = 1 | Total steps = 170
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


🏋️ Iniciando entrenamiento...
⏰ Tiempo estimado: ~20-30 min para 10k muestras en T4


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: alvarez-pablo-pa (alvarez-pablo-pa-openb) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.093300
2,1.150400
3,1.103000
4,1.060900
5,1.023400
6,1.040200
7,0.947900
8,0.932300
9,0.934700
10,0.918000


Step,Training Loss
1,1.093300
2,1.150400
3,1.103000
4,1.060900
5,1.023400
6,1.040200
7,0.947900
8,0.932300
9,0.934700
10,0.918000


🎉 ¡Entrenamiento completado!


## 🧪 Prueba del Modelo Fine-tuneado

Probemos el modelo con algunas preguntas para verificar su funcionamiento:

In [ ]:
# Preparar modelo para inferencia (2x más rápido)
model = FastLanguageModel.for_inference(model)

def test_model(prompt, max_tokens=128):
    """Función helper para probar el modelo"""
    messages = [{"from": "human", "value": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda" if torch.cuda.is_available() else "cpu")

    text_streamer = TextStreamer(tokenizer)
    print(f"🤖 Pregunta: {prompt}")
    print(f"💭 Respuesta: ", end="")

    _ = model.generate(
        input_ids=inputs,
        streamer=text_streamer,
        max_new_tokens=max_tokens,
        use_cache=True,
        temperature=0.7,
        do_sample=True
    )
    print("\n" + "="*50)

print("🧪 Probando el modelo fine-tuneado...")

🧪 Probando el modelo fine-tuneado...


In [ ]:
# Pruebas del modelo
test_prompts = [
    "¿Es 9.11 mayor que 9.9?",
    "Explica qué es el fine-tuning en términos simples",
    "¿Cuáles son las ventajas de usar LoRA?",
    "Escribe un código Python para calcular la secuencia de Fibonacci"
]

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n🔍 Prueba {i}/{len(test_prompts)}")
    test_model(prompt)

print("\n✅ Todas las pruebas completadas")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



🔍 Prueba 1/4
🤖 Pregunta: ¿Es 9.11 mayor que 9.9?
💭 Respuesta: <|im_start|>user
¿Es 9.11 mayor que 9.9?<|im_end|>
<|im_start|>assistant
1. The task is to compare two decimal numbers, 9.11 and 9.9, and determine which one is greater.
2. To compare decimal numbers, we need to convert them to a common form, such as a fraction or a decimal with the same number of decimal places.
3. In this case, we can compare 9.11 and 9.9 as decimal numbers.
4. Since 9.11 has one more digit to the right of the decimal point than 9.9, it is greater than 9.9.
5. Therefore, the correct answer is "Yes, 9


🔍 Prueba 2/4
🤖 Pregunta: Explica qué es el fine-tuning en términos simples
💭 Respuesta: <|im_start|>user
Explica qué es el fine-tuning en términos simples<|im_end|>
<|im_start|>assistant
El fine-tuning es un proceso de entrenamiento de un modelo de inteligencia artificial (IA) que se adapta a una tarea específica. Es una forma de entrenamiento más sofisticada en la que el modelo es entrenado para predecir o

## 💾 Guardar y Exportar el Modelo

Guardamos el modelo en diferentes formatos para máxima compatibilidad:

In [ ]:
# Opción 1: Guardar solo los adaptadores LoRA (más pequeño)
print("💾 Guardando adaptadores LoRA...")
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("✅ Adaptadores LoRA guardados en './lora_model'")

# Opción 2: Guardar modelo completo fusionado (16-bit)
print("\n🔗 Fusionando y guardando modelo completo...")
model.save_pretrained_merged("merged_model", tokenizer, save_method="merged_16bit")
print("✅ Modelo fusionado guardado en './merged_model'")

In [ ]:
# Opcional: Subir a Hugging Face Hub
# Descomenta y configura tu token de HF para subir el modelo

#from huggingface_hub import login
#login()  # Ingresa tu token de Hugging Face

# # Subir modelo fusionado
#model_name = "alvarezpablo/llama3.1-8b-finetune-sicyt-ar"
#model.push_to_hub_merged(model_name, tokenizer, save_method="merged_16bit")
#print(f"🚀 Modelo subido a: https://huggingface.co/{model_name}")

print("ℹ️ Para subir a HF Hub, descomenta y configura el código anterior")

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Unsloth: Merging weights into 16bit:   0%|          | 0/4 [00:00<?, ?it/s]

...

Unsloth: Merging weights into 16bit:  75%|███████▌  | 3/4 [07:54<02:37, 157.31s/it]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [08:29<00:00, 127.31s/it]


🚀 Modelo subido a: https://huggingface.co/alvarezpablo/llama3.1-8b-finetune-sicyt-ar
ℹ️ Para subir a HF Hub, descomenta y configura el código anterior


### 📦 Exportar a formato GGUF (para Ollama, LM Studio, etc.)

In [ ]:
# Exportar a GGUF para usar con Ollama, LM Studio, etc.
print("📦 Exportando a formato GGUF...")

# Diferentes niveles de cuantización
quant_methods = ["q4_k_m", "q5_k_m", "q8_0"]  # Métodos más comunes

for quant in quant_methods:
    print(f"🔄 Exportando {quant}...")
    model.save_pretrained_gguf(f"gguf_model", tokenizer, quantization_method=quant)
    print(f"✅ {quant} guardado")

print("\n🎉 Todos los formatos GGUF exportados en './gguf_model'")
print("💡 Puedes usar estos archivos con:")
print("   • Ollama: ollama create mi-modelo -f ./gguf_model")
print("   • LM Studio: Importar directamente")
print("   • llama.cpp: ./main -m ./gguf_model/model-q4_k_m.gguf")

## 🎯 Resumen y Próximos Pasos

### ✅ Lo que hemos logrado:
- Fine-tuning ultra-eficiente con **Unsloth** (2x más rápido, 60% menos memoria)
- Implementación de **Rank-Stabilized LoRA (rsLoRA)** para mejor estabilidad
- Uso del dataset **FineTome-100k** de ultra alta calidad
- Chat template **ChatML** optimizado
- Hiperparámetros basados en mejores prácticas
- Exportación a múltiples formatos (LoRA, merged, GGUF)

### 🚀 Próximos pasos sugeridos:
1. **Evaluación**: Usar Open LLM Leaderboard o LLM AutoEval
2. **Alignment**: Aplicar DPO con dataset de preferencias
3. **Cuantización**: Probar EXL2, AWQ, GPTQ para inferencia más rápida
4. **Deployment**: Usar Hugging Face Spaces, Ollama, o vLLM
5. **Escalado**: Probar con modelos más grandes (70B, 405B)

### 📚 Recursos adicionales:
- [LLM Course](https://github.com/mlabonne/llm-course) - Curso completo de LLMs
- [Unsloth Documentation](https://github.com/unslothai/unsloth) - Documentación oficial
- [FineTome Dataset](https://huggingface.co/datasets/mlabonne/FineTome-100k) - Dataset usado

---
 Módulo 4: Fine-tuning Optimizado 🇺🇾